# Decompress Lichess Puzzles `.zst`

Run this utility in Colab before `03a_build_strategy_datasets.ipynb` if your converted Lichess puzzle CSV is still compressed. It streams the compressed file to a plain `.csv` without loading the whole dataset into memory.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Paths

Expected input, with corrected spelling, is `Chess/data/strategy_sources/lichess_puzzles_converted.csv.zst`. This notebook also searches for common typo variants like `lichess_pouzzles_converted.csv.szt`.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")

SOURCE_DIR = PROJECT_ROOT / "data/strategy_sources"
OUTPUT_PATH = SOURCE_DIR / "lichess_puzzles_converted.csv"
OVERWRITE = False

candidate_names = [
    "lichess_puzzles_converted.csv.zst",
    "lichess_puzzles_converted.csv.szt",
    "lichess_pouzzles_converted.csv.zst",
    "lichess_pouzzles_converted.csv.szt",
    "lichess_db_puzzle.csv.zst",
]
candidate_paths = [SOURCE_DIR / name for name in candidate_names]

INPUT_PATH = next((path for path in candidate_paths if path.is_file()), None)
if INPUT_PATH is None:
    compressed = sorted(SOURCE_DIR.rglob("*.zst")) + sorted(SOURCE_DIR.rglob("*.szt"))
    if compressed:
        print("No default filename found. Compressed files discovered:")
        for path in compressed[:20]:
            print("-", path.relative_to(PROJECT_ROOT))
    raise FileNotFoundError(
        "Put the compressed puzzle file in data/strategy_sources/ as "
        "lichess_puzzles_converted.csv.zst, or edit INPUT_PATH manually."
    )

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)

## Install Decompressor

Uses the Python `zstandard` package so the file extension can be `.zst` or the mistyped `.szt`; the contents just need to be valid Zstandard.

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "zstandard"])
import zstandard as zstd
print("zstandard ready")

## Decompress

If the output file already exists, set `OVERWRITE = True` in the path cell and rerun.

In [ ]:
if OUTPUT_PATH.exists() and not OVERWRITE:
    raise FileExistsError(f"Output already exists: {OUTPUT_PATH}. Set OVERWRITE = True to replace it.")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
tmp_path = OUTPUT_PATH.with_suffix(OUTPUT_PATH.suffix + ".tmp")
dctx = zstd.ZstdDecompressor()
written = 0

with INPUT_PATH.open("rb") as compressed, tmp_path.open("wb") as output:
    with dctx.stream_reader(compressed) as reader:
        while True:
            chunk = reader.read(8 * 1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            written += len(chunk)
            if written and written % (256 * 1024 * 1024) < len(chunk):
                print(f"Wrote {written / (1024 ** 3):.2f} GiB...")

tmp_path.replace(OUTPUT_PATH)
print(f"Done: {OUTPUT_PATH}")
print(f"Decompressed bytes: {written:,}")

## Quick Preview

This prints the first few CSV rows so you can confirm the file is readable.

In [ ]:
import csv

with OUTPUT_PATH.open("r", encoding="utf-8", errors="replace", newline="") as stream:
    reader = csv.reader(stream)
    for index, row in zip(range(5), reader):
        print(row[:12])

print("Use this path in configs/strategy.yaml:")
print("data/strategy_sources/lichess_puzzles_converted.csv")